In [1]:
import pandas as pd
import seaborn as sns
import pathlib
import matplotlib.pyplot as plt
from collections import defaultdict

sns.set_style("whitegrid")
sns.set_theme(rc={
    "axes.facecolor": "white",     # axes (plot area)
    "figure.facecolor": "white",   # figure (canvas)
    "grid.color": "0.9",            # light gray grid
    "figure.figsize": (6, 3)
})

In [2]:
for file in pathlib.Path('..').glob('inspect.all_attribs.*.csv'):
    print('boxplot.' + file.with_suffix('.pdf').name.split('.', 2)[-1])
    df = pd.read_csv(file)
    df = df[df.correct].sample(100000)
    g = sns.boxplot(data=df, y='attrib', x='gold_label', showfliers=False)
    g.set(title=file.name)
    # sns.move_legend(g, "upper left", bbox_to_anchor=(1, 1))
    g.figure.savefig('boxplot.' + file.with_suffix('.pdf').name.split('.', 2)[-1], bbox_inches='tight')
    plt.clf()

boxplot.bcms_setimes.ig.pdf
boxplot.scandinavian.shap.pdf
boxplot.estonian_voro.lime.pdf
boxplot.bcms_setimes.lime.pdf
boxplot.estonian_voro.ig.pdf
boxplot.scandinavian.ig.pdf
boxplot.bcms_setimes.loo.pdf
boxplot.estonian_voro.shap.pdf
boxplot.scandinavian.loo.pdf
boxplot.scandinavian.lime.pdf
boxplot.estonian_voro.loo.pdf
boxplot.bcms_setimes.shap.pdf


<Figure size 600x300 with 0 Axes>

In [5]:
for grp in ['scandinavian', 'estonian_voro']:
    whitelists = defaultdict(set)
    with open(f'../data/{grp}/filtered_blacklist.txt', 'r') as istr:
        blacklist = map(str.strip, istr)
        # blacklist = map(str.lower, blacklist)
        blacklist = set(blacklist)
    with open(f'../data/{grp}/filtered_whitelist.txt', 'r') as istr:
        whitelist = map(str.strip, istr)
        # whitelist = map(str.lower, whitelist)
        for label, token in (record.split('\t') for record in whitelist):
            whitelists[label].add(token)
    for label in whitelists:
        assert len(whitelists[label] & blacklist) == 0, f'{label}\n{whitelists[label] & blacklist}'
        for labelB in whitelists:
            if labelB != label:
                assert len(whitelists[label] & whitelists[labelB]) == 0, f'{label}/{labelB}'
    def get_list(row):
        tok = row['token']
        if tok in blacklist:
            return 'falselisted'
        for label in whitelists:
            if tok in whitelists[label]:
                return 'truelisted (' + ('OK)' if label == row['gold_label'] else 'KO)')
        return 'unlisted'

    for file in pathlib.Path('..').glob(f'inspect.all_attribs.{grp}.*.csv'):
        print('boxplot_with_blacklist.' + file.with_suffix('.pdf').name.split('.', 2)[-1])
        df = pd.read_csv(file)
        df = df[df.correct].sample(100000)
        df['list'] = df.apply(get_list, axis=1)
        g = sns.boxplot(
            data=df, 
            y='attrib', 
            x='gold_label', 
            hue='list', 
            hue_order=['truelisted (OK)', 'unlisted', 'falselisted', 'truelisted (KO)'], 
            showfliers=False,
        )
        g.set(title=file.name)
        sns.move_legend(g, "upper left", bbox_to_anchor=(1, 1))
        g.figure.savefig('boxplot_with_list_status.' + file.with_suffix('.pdf').name.split('.', 2)[-1], bbox_inches='tight')
        plt.clf()
        g = sns.countplot(
            data=df, 
            # y='attrib', 
            x='gold_label', 
            hue='list', 
            hue_order=['truelisted (OK)', 'unlisted', 'falselisted', 'truelisted (KO)'], 
            dodge=True,
        )
        # Add labels to each bar
        for c in g.containers:
            g.bar_label(c, rotation=90)
        g.set(title=file.name, ylim=(0,30_000))
        sns.move_legend(g, "upper left", bbox_to_anchor=(1, 1))
        g.figure.savefig('histplot_with_list_status.' + file.with_suffix('.pdf').name.split('.', 2)[-1], bbox_inches='tight')
        plt.clf()

boxplot_with_blacklist.scandinavian.shap.pdf
boxplot_with_blacklist.scandinavian.ig.pdf
boxplot_with_blacklist.scandinavian.loo.pdf
boxplot_with_blacklist.scandinavian.lime.pdf
boxplot_with_blacklist.estonian_voro.lime.pdf
boxplot_with_blacklist.estonian_voro.ig.pdf
boxplot_with_blacklist.estonian_voro.shap.pdf
boxplot_with_blacklist.estonian_voro.loo.pdf


<Figure size 600x300 with 0 Axes>

In [4]:


df = []
for xai in ['ig', 'loo', 'lime', 'shap']:
    df.append(pd.read_csv(f"../inspect.all_attribs.bcms_setimes.{xai}.csv"))
    df[-1]['XAI'] = xai
    df[-1] = df[-1][df[-1].correct].sample(100000)
df = pd.concat(df)
df

ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [ ]:
g = sns.boxplot(data=df, y='attrib', x='XAI', hue='gold_label', showfliers=False)
g.set(ylim=(-1.25, 1.25))
sns.move_legend(g, "upper left", bbox_to_anchor=(1, 1))
g.figure.savefig('attrib_distrib.CS.pdf', bbox_inches='tight')

In [ ]:
df = []
for xai in ['ig', 'loo', 'lime', 'shap']:
    df.append(pd.read_csv(f"../inspect.all_attribs.estonian_voro.{xai}.csv"))
    df[-1]['XAI'] = xai
    df[-1] = df[-1][df[-1].correct].sample(100000)
df = pd.concat(df)
df

In [ ]:
g = sns.boxplot(data=df, y='attrib', x='XAI', hue='gold_label', showfliers=False)
g.set(ylim=(-1.25, 1.25))
sns.move_legend(g, "upper left", bbox_to_anchor=(1, 1))
g.figure.savefig('attrib_distrib.EV.pdf', bbox_inches='tight')

In [ ]:
df = []
for xai in ['ig', 'loo', 'lime', 'shap']:
    df.append(pd.read_csv(f"../inspect.all_attribs.scandinavian.{xai}.csv"))
    df[-1]['XAI'] = xai
    df[-1] = df[-1][df[-1].correct].sample(100000)
df = pd.concat(df)
df

In [ ]:
g = sns.boxplot(data=df, y='attrib', x='XAI', hue='gold_label', showfliers=False)
g.set(ylim=(-1.25, 1.25))
sns.move_legend(g, "upper left", bbox_to_anchor=(1, 1))
g.figure.savefig('attrib_distrib.scandi.pdf', bbox_inches='tight')